In [1]:
import pennylane as qml
from pennylane import numpy as np
import networkx as nx

In [3]:
edge = [[0,1],[0,3],[1,2],[1,4],[2,3],[4,5],[4,6],[5,6],[6,7],[6,10],[7,8],[7,9],[7,10]]
graph = nx.Graph(edge)
cost_hamil,mixer_hamil = qml.qaoa.maxcut(graph)
def qaoa_layer(gamma,beta):
    qml.qaoa.cost_layer(gamma, cost_hamil)
    qml.qaoa.mixer_layer(beta, mixer_hamil)

In [4]:
num = len(graph.nodes)
device = qml.device('default.qubit',wires=num)
def quantum_circuit(params):
    gamma = params[0]
    beta = params[1]
    for q in range(num):
        qml.Hadamard(wires=q)
    p=len(gamma)
    for i in range(p):
        qaoa_layer(gamma[i],beta[i])
    return qml.expval(cost_hamil)

In [5]:
circuit = qml.QNode(quantum_circuit,device)
p = 5
params = np.random.uniform(0,np.pi,(2,p),requires_grad = True)
optimiser = qml.GradientDescentOptimizer(stepsize=0.3)
for i in range(30):
    params,cost_computed = optimiser.step_and_cost(circuit,params)
    if(i%5==0):
        print(f'current parameter is {params} an cost is {cost_computed}')
print(f'final paramter is {params}')

current parameter is [[ 3.03887612  1.24931491  2.21348653  2.48274833  2.50258718]
 [ 2.19603407  0.66222817  0.73531735 -0.03218805  1.97280322]] an cost is -8.095808341225435
current parameter is [[ 3.82148418  0.91210568  1.65644526  3.75670082  3.99064187]
 [ 2.44241677 -0.66882223  2.69587073  0.14579155  4.54124604]] an cost is -7.956591868856952
current parameter is [[4.4373288  1.69494277 1.37787137 4.40437008 3.97898095]
 [1.21814678 1.1087333  4.09994259 0.88703463 3.97334917]] an cost is -6.441149465499943
current parameter is [[4.24841215 1.70792602 1.76875865 4.5421359  4.12904192]
 [0.8609502  0.88809622 3.96640002 0.46372832 3.63043928]] an cost is -6.584067964925537
current parameter is [[3.8377301  1.86959796 1.59689049 4.6362686  4.6765209 ]
 [1.77687876 1.06240325 4.22067903 0.575974   4.41250505]] an cost is -6.652897652269965
current parameter is [[ 4.13182966e+00  2.19595584e+00  2.09182921e+00  3.90194150e+00
   4.09476125e+00]
 [ 1.24494783e+00 -2.84356473e-03 

In [6]:
@qml.qnode(device)
def probability(params):
    gamma = params[0]
    beta = params[1]
    for i in range(num):
        qml.Hadamard(wires=i)
    for i in range(len(gamma)):
        qaoa_layer(gamma[i],beta[i])
    return qml.probs(wires=range(num))

In [7]:
probabilities = probability(params)
print(probabilities)

[0.00103717 0.00013046 0.00043835 ... 0.00043835 0.00013046 0.00103717]


In [9]:
best = np.argmax(probabilities)
best_string = format(best,'011b')
print(best_string)

00010001000


In [10]:
A = []
B = []
for i in range(len(best_string)):
    if best_string[i]=='0':
        A.append(i)
    else:
        B.append(i)
print(f'set A has {A}')
print(f'set B has {B}')

set A has [0, 1, 2, 4, 5, 6, 8, 9, 10]
set B has [3, 7]
